# 01_02_Collecte de données depuis le portail Open Data de Statbel #

In [1]:
from pathlib import Path
from zipfile import ZipFile

import geopandas as gpd
import pandas as pd

import infrabel_punctuality as ip

# Configurer le chemin du répertoire de destination
DATA_DIR = Path("../data/raw/")


## 1. Dictionnaires des datasets à importer depuis l'Open Data de Statbel et geo.be ##

Dans la data warehouse sous SQL Server, la dimension commune sera "snowflakée" (fact_punctuality -> dim_stations -> dim_communes) pour éviter la redondance des communes (plusieurs gares pour une même commune) et pour structurer la hiérarchie géographique commune -> province -> région. Dans le modèle sémantique sur Power BI, la dimension commune sera dénormalisée et intégrée à la dimension gares (star schema).

### 1.1. Dictionnaire 1 : datasets au format xlxs ##

* dataset "communes" : future dimension commune.
* dataset "population" : données destinées à enrichir la dimension commune pour analyses des retards en fonction de l'environnement urbain (grandes villes, villes moyennes, communes rurales).

In [2]:
datasets_3 = {
    "communes" : "https://statbel.fgov.be/sites/default/files/files/opendata/REFNIS%20code/TU_COM_REFNIS-20250101.xlsx",
    "population" : "https://statbel.fgov.be/sites/default/files/files/documents/bevolking/5.11%20Bevolkingsdichtheid/Pop_density_fr.xlsx"
}

### 1.2. Dictionnaire 2 : datasets pour la geolocalisation depuis geo.be ###

* dataset "geocommunes" : données destinées à enrichir la dimension commune pour permettre la cartographie de données.</br>
    Dossier compressé zip. Format geopackage (.gpkg) avec projection de Lambert 2008 (EPSG:3812).</br>
    Source : geo.be est la plateforme géographique de l'État fédéral belge. Ce dataset est en licence creativecommons 4.0.

In [3]:
datasets_4 = {
    "geolocalisation" : "https://ac.ngi.be/remoteclient-open/ngi-standard-open/Vectordata/TerritorialDivisions/TerritorialDivisions-All/1cd00540-b0ca-11ec-a863-186571a04de2_geopackage+sqlite3_3812.zip"
}


## 2. Téléchargement des datasets ##

### 2.1. Configuration des paramètres et téléchargement ###

In [4]:
# Configuration du type de fichier et du nom du manifeste de téléchargement : file_type="xlxs", registry_name="manifest_statbel.txt"
ip.run_download(datasets_3, output_dir=DATA_DIR, file_type="xlsx", registry_name="manifest_statbel.txt")

Début du téléchargement : communes



communes téléchargé dans ..\data\raw\communes.xlsx
Début du téléchargement : population



population téléchargé dans ..\data\raw\population.xlsx


In [5]:
# Configuration du type de fichier et du nom du manifeste de téléchargement : file_type="zip", registry_name="manifest_statbel.txt"
ip.run_download(datasets_4, output_dir=DATA_DIR, file_type="zip", registry_name="manifest_geo.txt")

Début du téléchargement : geolocalisation



geolocalisation téléchargé dans ..\data\raw\geolocalisation.zip


### 2.2. Décompression des données géographiques ###

In [6]:
# Configurer le chemin vers le dossier zip
zip_path = Path("geolocalisation.zip")

with ZipFile(DATA_DIR / zip_path, "r") as zip_ref:
    zip_ref.extractall(DATA_DIR / zip_path.stem)

In [7]:
final_path = DATA_DIR / zip_path.stem

# afficher la liste des fichiers décompressés
for file in final_path.iterdir():
    print(file.name)

territorialdivisions_3812.gpkg


### 2.3. Vérification du téléchargement par lecture des fichiers ###

In [8]:
df = pd.read_excel(DATA_DIR / "communes.xlsx")
df.head()

,LVL_REFNIS,CD_REFNIS,CD_SUP_REFNIS,TX_REFNIS_DE,TX_REFNIS_FR,TX_REFNIS_NL,DT_VLDT_START,DT_VLDT_END
0,1,2000,-,Flämische Region,Région flamande,Vlaams Gewest,01/01/1970,31/12/9999
1,1,3000,-,Wallonische Region,Région wallonne,Waals Gewest,01/01/1970,31/12/9999
2,1,4000,-,Region Brüssel-Hauptstadt,Région de Bruxelles-Capitale,Brussels Hoofdstedelijk Gewest,01/01/1970,31/12/9999
3,2,10000,02000,Provinz Antwerpen,Province d’Anvers,Provincie Antwerpen,01/01/1970,31/12/9999
4,2,20000,-,Provinz Brabant,Province de Brabant,Provincie Brabant,01/01/1970,31/12/1994


In [9]:
# 1re colonne du tableau excel = titre du tableau -> préciser que les titres des colonnes sont la 2e ligne (header=1)
df = pd.read_excel(DATA_DIR / "population.xlsx", header=1)
df.head()

,Refnis,Lieu de résidence,Population,Superficie au km²,Habitants / km²
0,1000,Belgique,11431406,30689.368862,372.487491
1,2000,Région flamande,6589069,13625.548930,483.581912
2,3000,Région wallonne,3633795,16901.400088,214.999644
3,4000,Région de Bruxelles-Capitale,1208542,162.419844,7440.851870
4,10000,Province d’Anvers,1857986,2876.120363,646.004258


In [10]:
gdf = gpd.read_file(final_path / "territorialdivisions_3812.gpkg", layer="municipality")
gdf.head()

,tgid,modifdate,arrondissementcapital,provincecapital,regioncapital,countrycapital,niscode,city,languagestatute,nameger,namefre,namedut,geometry
0,{8BF44CB0-B8FD-44F6-A64F-1307610DA4C9},2007-01-05,False,False,False,False,72004,1,1,Bree,Bree,Bree,"MULTIPOLYGON Z (((735277.943 700725.863 0, 735..."
1,{54A85359-4967-4318-AA63-D234DDED2FD7},2007-01-05,False,False,False,False,63004,0,2,Baelen,Baelen,Baelen,"MULTIPOLYGON Z (((761804.571 647035.961 0, 761..."
2,{95E2AAE2-F9DB-456C-A113-2A21ED6F932F},2007-01-05,False,False,False,False,13003,0,1,Balen,Balen,Balen,"MULTIPOLYGON Z (((708505.572 703629.811 0, 708..."
3,{4487B4B8-4422-4856-97C5-33174BF84028},2007-01-05,False,False,False,False,62011,0,2,Bassenge,Bassenge,Bitsingen,"MULTIPOLYGON Z (((736728.393 660624.972 0, 736..."
4,{68A224E9-B2E7-4225-9FDA-03AE2B9C8C41},2007-01-05,False,False,False,False,85046,0,2,Habay,Habay,Habay,"MULTIPOLYGON Z (((735041.579 544346.475 0, 734..."
